In [0]:
# Databricks Notebook: 02_ingest_kworb
import requests
import os
from datetime import datetime

# Parameter Widget for Backfilling
dbutils.widgets.text("process_date", "", "Process Date (YYYY-MM-DD)")
param_date = dbutils.widgets.get("process_date").strip()

# Default to current UTC date if widget is empty
today = param_date if param_date else datetime.utcnow().strftime("%Y-%m-%d")

GITHUB_USER = "mahar3b"
REPO_NAME = "Spotiflow"
BRANCH = "main"

VOLUME_PATH = f"/Volumes/workspace/spotipulse_db/raw_data/kworb/{today}"
os.makedirs(VOLUME_PATH, exist_ok=True)

regions = ["global", "us", "gb", "de", "jp", "br", "ca", "au", "fr", "mx"]
print(f"--- Processing Kworb Ingestion for Date: {today} ---")

downloaded_count = 0
for region in regions:
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/data/kworb/{today}/{region}.csv"
    resp = requests.get(url)
    
    if resp.status_code == 200:
        target_file = f"{VOLUME_PATH}/{region}.csv"
        with open(target_file, "wb") as f:
            f.write(resp.content)
        print(f"✔ Downloaded: {region} -> {target_file}")
        downloaded_count += 1
    else:
        print(f"✖ Skipping {region}: File not available on GitHub (HTTP {resp.status_code})")

print(f"\nCompleted: {downloaded_count}/{len(regions)} region CSVs stored in Volume.")